In [ ]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad

In [ ]:
# experimental data
save_folder = 'run7'
n_points = 5000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [ ]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [ ]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q2, phi, mg, a1, a2, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q2, phi, mg, a1, a2, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  


def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [ ]:
n_points = 1750

def full_int(mg, a1, a2, m2_func, q2_val, sqrt_s):
    # Garante que q_val seja array 1D
    q2_val = np.atleast_1d(q2_val)
    results = []

    for q2 in q2_val:
        def integrand(y, x, mg, a1, a2, m2_func, q2_val):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            result = k * (
                T_1(k, q2_val, phi, mg, a1, a2, m2_func)
                - T_2(k, q2_val, phi, mg, a1, a2, m2_func)
            ) * jacobian
            return result

        def inner_integral(x):
            integral_real = fixed_quad(
                lambda y: np.real(integrand(y, x, mg, a1, a2, m2_func, q2)),
                0, 1, n=n_points
            )[0]
            integral_imag = fixed_quad(
                lambda y: np.imag(integrand(y, x, mg, a1, a2, m2_func, q2)),
                0, 1, n=n_points
            )[0]
            return integral_real + 1j * integral_imag

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]

In [ ]:
eps_min = 0.094763
mg_min = 0.93
a1_min = 1.4	
a2_min = 2.69

In [ ]:
# Calculates and plot dif sigma 
lst_amp_born_diff = []
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 1e-2
    max_q2   = 0.2001
    q2_step  = 0.001


    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2
        integral_value = full_int(
                mg, a1, a2, mg_model, q2, sqrt_s
        )
        # print(integral_value)
        
        diff_T = integral_value


        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        lst_amp_born_diff.append(amp_value)
        dif_sigma  = differential_sigma(amp_value, s) * scale
        # print(f"q2 = {q2}, diff_t = {integral_value}, amp born = {amp_value:6e} \n")

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

In [ ]:
import numpy as np
from scipy.special import j0
from scipy.integrate import quad

lst_chi = []
lst_amp_eik = []
lst_diff_sigma = []


# sqrt_s = 7000
# s = sqrt_s ** 2

eps_rel = 1e-6
eps_abs = 1e-16

q_max_chi = 5.0          # limite de q na Eq. 23
b_max     = 20.0

limit = 1000

# ── Eq. 23: χ(s,b) como integral direta em q ─────────────────────────────────
def chi(b_val, mg_model, sqrt_s, eps, mg, a1, a2):
    def integrand_real(q_val):
        q2_val = q_val ** 2
        t = -q2_val
        s = sqrt_s ** 2                                          # corrigido
        diff_t = full_int(mg, a1, a2, mg_model, q2_val, sqrt_s) # corrigido
        born_amp = amp_calculation(diff_t, s, eps, t)            # corrigido
        return np.real((1/s) * q_val * j0(b_val * q_val) * born_amp)

    def integrand_imag(q_val):
        q2_val = q_val ** 2
        t = -q2_val
        s = sqrt_s ** 2                                          # corrigido
        diff_t = full_int(mg, a1, a2, mg_model, q2_val, sqrt_s) # corrigido
        born_amp = amp_calculation(diff_t, s, eps, t)            # corrigido
        return np.imag((1/s) * q_val * j0(b_val * q_val) * born_amp)

    real_part, _ = quad(integrand_real, 0, q_max_chi, epsrel=eps_rel, epsabs=eps_abs, limit=limit)
    imag_part, _ = quad(integrand_imag, 0, q_max_chi, epsrel=eps_rel, epsabs=eps_abs, limit=limit)
    return real_part + 1j * imag_part

In [ ]:
from scipy.interpolate import interp1d

# Fixed integration grids (defined once globally)
Q_GRID = np.linspace(0, q_max_chi, 300)     # q points for chi integral
B_GRID = np.linspace(0, b_max, 200)         # b points for eikonal integral

def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):
    m2 = m2_log
    s = sqrt_s ** 2

    # ── Step 1: precompute Born amplitude on Q_GRID ──
    born_on_grid = np.array([
        amp_calculation(full_int(mg, a1, a2, m2, q**2, sqrt_s), s, eps, -q**2)
        for q in Q_GRID
    ])

    # ── Step 2: build chi(b) on B_GRID ──
    def chi_fast(b_val):
        integrand_vals = (1/s) * Q_GRID * j0(b_val * Q_GRID) * born_on_grid
        real_part = np.trapezoid(np.real(integrand_vals), Q_GRID)
        imag_part = np.trapezoid(np.imag(integrand_vals), Q_GRID)
        return real_part + 1j * imag_part

    chi_on_grid = np.array([chi_fast(b) for b in B_GRID])

    # ── Step 3: compute eikonal amplitude for each data point q² ──
    lst_diff_sigma_eik = []
    for q2 in x:
        q_ext = np.sqrt(q2)
        integrand_eik = B_GRID * j0(q_ext * B_GRID) * (1 - np.exp(1j * chi_on_grid))
        
        # Use trapezoid instead of deprecated trapz
        amp_eik = 1j * s * np.trapezoid(integrand_eik, B_GRID)
        
        # Correct differential cross-section: dσ/dt = (π/s²) * |A|²
        # with conversion factor 0.389379323 mb·GeV²
        amp_squared = np.abs(amp_eik)**2
        diff_sigma = (np.pi / s**2) * amp_squared * 0.389379323
        lst_diff_sigma_eik.append(diff_sigma)

    return np.array(lst_diff_sigma_eik)

In [ ]:

def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='pl')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='pl')
    
def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='pl')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7, verbose=True)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8, verbose=True)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13, verbose=True)

chi2_total = chi2_7

# chi2_total = chi2_7 

minuit_eik = Minuit(
    chi2_total,
    mg = mg_min,
    a1 = a1_min,
    a2 = a2_min,
    eps = eps_min,
)


minuit_eik.limits['mg']  = (0.1, 2.0)
minuit_eik.limits['a1']  = (0.1, 10.0)
minuit_eik.limits['a2']  = (0.1, 10.0)   # a2 must be positive (Gaussian form factor)
minuit_eik.limits['eps'] = (0.0, 2)

ncall = 10000

minuit_eik.simplex(ncall=ncall)
minuit_eik.migrad(ncall=ncall)
minuit_eik.hesse()
